# Train balance bot using PPO with domain randomization - 6 observations (CPU) - Sim2Real improved

Eight phases preserved. Updates: encoder 414 ticks/rev, wheel-speed calculation over 25 ms, independently randomized motor gains, position/speed rewards for mean and difference, gamma=0.999. **Warning:** Complementary IMU estimator is still not the same as on-device Kalman. Motor torque/deadzone, the exact MJCF time step and servo dynamics remain to be identified; do not claim sim2real equivalence from these changes alone.

**V2:** eight stages; last stage consolidates motor response, no gearbox backlash training. Measured no-load curves determine *conditional* gear/friction parameters. Full motor/IMU model still needs loaded tests and Kalman equivalence.


**CPU edition:** no CUDA/ROCm required.


In [ ]:
# Import standard libraries
from dataclasses import replace
import os
from pathlib import Path
import sys
import time

# Third-party libraries
import gymnasium as gym
from gymnasium.wrappers import RecordEpisodeStatistics
import numpy as np
import torch

# Import custom environment
from balance_bot_env_dr_8_phases import BalanceBotEnv, DomainRandomConfig

# Add the folder containing our envs/ and rl/ packages to the path
sys.path.append("/workspace/software")

# Import PPO training module and exporter
from rl.ppo_trainer import PPOConfig, evaluate, train, export_tb_plots_as_csv, export_actor_onnx
from rl.onnx_actor_to_c import export_onnx_actor_to_c

In [ ]:
# CPU-only training. Explicitly disable CUDA/ROCm, even if available.
CPU_DEVICE = torch.device("cpu")
CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)
print(f"PyTorch {torch.__version__} | device={CPU_DEVICE} | threads={CPU_THREADS}")

def train_on_cpu(config, **kwargs):
    if config.cuda:
        raise ValueError("CPU training requires PPOConfig.cuda=False")
    result = train(config, **kwargs)
    if next(result.agent.parameters()).device.type != "cpu":
        raise RuntimeError("PPO agent is not running on the CPU")
    return result


In [ ]:
# Settings -- full GPU-equivalent training budget, executed exclusively on CPU.
MJCF_PATH = Path("/workspace/mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42
NUM_ENVS = 8               # Same number of parallel environments as GPU version
STEPS_PER_ENV = 500_000     # Same number of steps per environment and phase as GPU version


## Configure PPO and environment

In [ ]:
# Configure PPO
ppo_config = PPOConfig(
    cuda = False,                  # CPU-only PPO
    exp_name = "balance-bot6obs-ppo-cpu",  # Name of the experiment
    env_id = "BalanceBot-v0",      # Name of the environment
    seed = SEED,                   # Constant seed for reproducibility
    num_envs = NUM_ENVS,           # Number of parallel environments
    actor_hidden_layers = 2,       # Number of hidden layers in the actor network
    actor_hidden_size = 32,        # Number of nodes in each hidden layer in the actor
    critic_hidden_layers = 2,      # Number of hidden layers in the critic network
    critic_hidden_size = 32,       # Number of nodes in each hidden layer in the critic
    total_timesteps = NUM_ENVS * STEPS_PER_ENV,  # Total simulation steps (all envs and iterations)
    num_steps = 2048,              # Same rollout length as GPU version (10.24 s per env)
    num_minibatches = 32,          # Same minibatch count as GPU version
    update_epochs = 10,            # Number of epochs to update actor and critic for each iteration
    anneal_lr = True,              # Enable annealing (lower learning rate as training goes on)
    learning_rate = 3e-4,          # Initial learning rate, reduced by annealing (if enabled)
    gamma = 0.999,                  # Discount factor (future rewards are discounted by this amount)
    gae_lambda = 0.97,             # GAE blending: 0 = pure TD error, 1 = pure Monte Carlo
    clip_coef = 0.2,               # Limits policy ratio to prevent large actor updates
    value_clip = 1.0,              # Absolute bounds on value prediction change per update (critic)
    ent_coef = 0.0,                # How much entropy factors into total loss calculation
    vf_coef = 0.5,                 # How much the value loss factors into total loss calculation
    max_grad_norm = 0.5,           # Limits how much actor/critic parameters can change during an update
    checkpoint_interval = 50,      # Save model every 50 iterations
    save_model = True,             # Save the final model
    timestep = 0.000,              # Match MJCF opt.timestep for real-time rendering (or 0 for fast)
)

In [ ]:
def make_balance_bot_env(render, **kwargs):
    """Function to create an environment for our balance bot"""
    # Create the environment and set the render mode
    env = BalanceBotEnv(
        mjcf_path    = MJCF_PATH,
        render_mode  = "human" if render else None,
        **kwargs
    )

    # Wrap in RecordEpisodeStatistics so we can log episodic returns in the 'info' dict
    return gym.wrappers.RecordEpisodeStatistics(env)

def make_envs(num_envs, **kwargs):
    """Create a SyncVectorEnv with num_envs balance bot environments."""
    env_factories = []
    for i in range(num_envs):
        env_factories.append(
            # Uncomment this and comment the next line to preview the first environment:
            # lambda render=(i==0), kw=kwargs: make_balance_bot_env(render, **kw)
            lambda render=False, kw=kwargs: make_balance_bot_env(render, **kw)
        )
    return gym.vector.SyncVectorEnv(env_factories)

In [ ]:
def load_agent(envs, model_path):
    """
    For debugging only! Use this to load a previously trained model to skip previous phases. Note 
    that you will still need to run the cells in each prior phase that update the experiment name 
    and environment.
    """
    from rl.ppo_trainer import Agent, TrainResult

    # Make sure model_path is a Path
    model_path = Path(model_path)
    
    # Load agent from previous run
    device = CPU_DEVICE
    agent = Agent(envs, ppo_config).to(device)
    agent.load_state_dict(torch.load(model_path, map_location=device, weights_only=True))
    agent.eval()
    print(f"Loaded model from {model_path}")
    
    # Wrap agent in a dummy result
    return TrainResult(
        agent = agent,
        checkpoint_dir = model_path.parent,
        best_model_path = model_path,
        final_model_path = None,
        best_mean_return = 0,
    )

## Phase 1: Balance only

We'll start with the easy task of only penalizing if the robot tilts (pitches) forward.

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-1-cpu"

# Create an environment that only rewards staying upright
envs = make_envs(
    NUM_ENVS,
    pitch_penalty_coef=0.5,
    action_penalty_coef=0.01,
    position_penalty_coef=0.0,
    wheel_position_penalty_coef=0.0,
    yaw_penalty_coef=0.0
)
# Fail fast if the XML runs at a different physical time step than firmware.
# Otherwise the 5-sample encoder window and PPO discount horizon do not match.
for wrapper in envs.envs:
    sim_dt = float(wrapper.env.model.opt.timestep)
    if abs(sim_dt - 0.005) > 1e-6:
        raise RuntimeError(
            f"MJCF timestep={sim_dt}s; expected 0.005s (200 Hz) as on BALA2. "
            "Update the <option timestep=...> in the MJCF and revalidate physics."
        )

assert envs.single_observation_space.shape == (6,), "Expected six observations"
assert envs.single_action_space.shape == (2,), "Expected two motor actions"


In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 2: Penalize position and rotation

We start with the agent (actor and critic networks) trained in phase 1. We then update the existing environments to apply a penalty for any motion off the origin as well as rotation around the Z axis (yaw). We then train the existing agent further in this new environment.

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-2-cpu"

# Update the position and rotation coefficients in the existing environments
for env_stat_wrapper in envs.envs:
    env = env_stat_wrapper.env
    env.position_penalty_coef = 0.001
    env.wheel_position_penalty_coef = 0.004  # Discourage backward/forward drift
    env.wheel_difference_penalty_coef = 0.004  # Discourage accumulated turning
    env.wheel_mean_velocity_penalty_coef = 0.001  # Discourage continuous rolling
    env.wheel_difference_velocity_penalty_coef = 0.001  # Discourage spinning
    env.yaw_penalty_coef = 0.1

In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 3: Observation noise and action delay

With the robot staying upright and mostly in place, we add the additional difficulty of randomizing the observations (injecting noise) and introducing a random 0..1 timestep delay of how long it takes the chosen action to take effect.

The noisy observations model how real IMUs work. You will often find some noise in their readings, and it can help the robot deal with biases from an imperfectly calibrated IMU.

The action delay models the time it takes for the robot to respond to a chosen action. In simulation, a chosen action takes effect on the very next `step()` call. In real hardware, it could take up to a few milliseconds, thanks to extra code, communication bus latency (e.g. I2C), and motor lag.

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-3-cpu"

# Create data randomization config to add obs noise and action delay
dr = DomainRandomConfig(
    pitch_noise_std_dev=0.01,       # Inject some noise into pitch observation
    pitch_rate_noise_std_dev=0.01,  # Inject some noise into pitch rate observation
    wheel_vel_noise_std_dev = 0.1,  # Inject some noise into wheel velocity observation
    wheel_pos_ticks_per_rev=414.0,  # Hall encoder quantization at wheel output
    wheel_pos_noise_std_dev=0.005,
    encoder_velocity_window_steps=5, # same 25 ms as ESP32 # Small extra angular observation noise (rad)
    action_delay_steps=1,           # 1 step (5ms) delay
    action_delay_random=True,       # Vary 0-1 steps each episode
)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# DEBUG: Uncomment this cell to load a previously saved model
# result = load_agent(envs, "runs/BalanceBot-v0__balance-bot8-phase-2__42__1778905085/best_model.cleanrl_model")

In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 4: Add motor noise and random pushes

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-4-cpu"

# Add motor noise and pushes to our domain randomization config
dr.motor_noise_scale = 0.02 # +/-2% motor noise
dr.push_prob = 0.005        # 0.5% chance of push per step
dr.push_force_max_n = 0.3   # Gentle pushes

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 5: Add mass and friction randomization

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-5-cpu"

# Add mass and friction randomization to the DR config
dr.mass_scale_range= (0.8, 1.2)       # +/-20% mass variation
dr.friction_scale_range = (0.5, 1.5)  # 50-150% friction variation

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 6: Add motor gain randomization

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-6-cpu"

# Add motor gain randomization to our DR config
dr.motor_gain_range = (0.90, 1.10)    # around nominal motor curves; provisional tolerance, not identified
dr.independent_motor_gains = True  # left/right gains independently sampled

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 7: Add random axle torques (simulate tire ridges)

In [ ]:
# Update experiment name
ppo_config.exp_name = "6obs-balance-bot8-phase-7-cpu"

# Add random axle torques to simulate tire ridges
dr.ridge_prob=0.05                  # 5% possibility of tire "ridge"
dr.ridge_torque_max_nm=0.005        # Add a slight torque to the joints (simulate tire ridges)

# Update the domain randomization in the environments
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr

In [ ]:
# Choo choo train
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)

# Inspect what was saved
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Best mean return: {result.best_mean_return:.2f}")

# Load best model if available, otherwise use final
if result.best_model_path is not None:
    result.agent.load_state_dict(
        torch.load(result.best_model_path, map_location=CPU_DEVICE, weights_only=True)
    )
    print(f"Loaded best model (mean_return={result.best_mean_return:.2f})")

In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Phase 8: Consolidation of the Identified Motor Model (WITHOUT Backlash)

No-load PWM pulses at 100, 180, and 300 indicate an approximately linear speed response, with a slight left/right asymmetry.

The Python script **conditionally calibrates** `actuator_gear` and `frictionloss` based on these measurements, while preserving the joint damping defined in the MJCF model.

This **does not constitute a measurement of absolute motor torque** under load.

Backlash remains **disabled** (0-degree play), as the direction-reversal tests do not yet allow transmission backlash to be distinguished from inertia and electrical braking. It may be reintroduced later if dedicated measurements demonstrate its significance.

This phase consolidates the Phase 7 PPO policy using more moderate parameter variations around the measured characteristics of each motor.

It introduces **no corrections whatsoever** to the agent's predictions.

**Keep a Phase 7 model for A/B comparison.**

In [ ]:
# Phase 8: consolidation around empirically measured motor curves, NOT backlash.
ppo_config.exp_name = "6obs-bala2-v3-phase-8-motor-consolidation-cpu"
dr.backlash_gap_deg_range = (0.0, 0.0)  # Strictly off: not identified
# Keep independently randomized left/right motor performance.
dr.independent_motor_gains = True
dr.motor_gain_range = (0.95, 1.05)  # Focus near measured nominal, provisional
# Reduce impulsive axle disturbances for a consolidation experiment.
dr.ridge_prob = 0.02
dr.ridge_torque_max_nm = 0.002
# No extra jerk penalty without evidence of beneficial real behavior.
dr.action_delta_penalty_coef = 0.0
for env_stat_wrapper in envs.envs:
    env_stat_wrapper.env.dr = dr
obs, infos = envs.reset(seed=SEED)
assert all(w.env._backlash_gap_rad == 0.0 for w in envs.envs)
for wrapper in envs.envs[:1]:
    env = wrapper.env
    print('Calibrated nominal motor gains (L,R):',
          env._left_gear_orig, env._right_gear_orig)
    print('Calibrated nominal joint friction (L,R):',
          env.model.dof_frictionloss[env._left_wheel_dof_idx],
          env.model.dof_frictionloss[env._right_wheel_dof_idx])


In [ ]:
# PPO phase 8: transfer learning from phase 7
result = train_on_cpu(ppo_config, envs=envs, agent=result.agent)
print(f"Best model: {result.best_model_path}")
print(f"Final model: {result.final_model_path}")
print(f"Checkpoint folder: {result.checkpoint_dir}")


In [ ]:
# No real-time delay during evaluation (use 0.005 for real-time preview)
#eval_config = replace(ppo_config, timestep=0.005)

# Evaluate
#returns = evaluate(
#    result.agent, 
#    eval_episodes=3, 
#    config=eval_config, 
#    envs=envs)

# Print 
#print(f"Mean return: {np.nanmean(returns):.2f}")

# Get the run directory
#run_path = result.checkpoint_dir

# Export TensorBoard plots as CSV files
#export_tb_plots_as_csv(run_path)

## Clean up and save model

At this point, we are done with training. We want to delete the environments and save the best actor from our final training phase. We'll export this actor network as an ONNX file that can be used on a variety of hardware platforms.

In [ ]:
# Close the environments
for idx, env in enumerate(envs.envs):
    print(f"Closing env {idx}")
    env.env.close()

In [ ]:
# Get observation and action sizes
obs_size = envs.single_observation_space.shape[0]
action_size = envs.single_action_space.shape[0]
assert obs_size == 6, f"Expected 6 observations, got {obs_size}"
assert action_size == 2, f"Expected 2 actions, got {action_size}"

# Export the actor network as an ONNX model
export_actor_onnx(
    model_path=result.best_model_path,
    output_path=result.checkpoint_dir / "actor.onnx",
    obs_size=obs_size,
    action_size=action_size,
    num_hidden_layers=ppo_config.actor_hidden_layers,
    hidden_layer_size=ppo_config.actor_hidden_size,
)

In [ ]:
# Export actor to .h file
export_onnx_actor_to_c(
    onnx_path   = result.checkpoint_dir / "actor.onnx",
    output_path = result.checkpoint_dir / "actor.h"
)